## Notebook 14 — Perbaikan Finding 10 + Perluasan Finding 09 ke 6 Tipe

Dua bagian independen, jalan berurutan di 1 sesi (model dimuat sekali):

**Part A (perbaikan finding 10):** notebook 13 memilih soal PER PASANGAN
(top-disagreement 2 kelompok tertentu) -> median cuma 2-3 kelompok/soal,
nggak cukup buat probe. Di sini soal dipilih pakai **cakupan luas**
(dijawab banyak/semua sel per tipe, logika sama dengan notebook 07/09),
lalu ekstrak vektor **L11 DAN L1** (L1 = kandidat baru dari finding 09
buat RACE) di posisi jawaban-opini, buat SEMUA 6 tipe.

**Part B (perluasan finding 09):** sweep 32-layer + kontrol acak (mesin
yang sama persis dengan notebook 13, sudah diverifikasi lolos 5+3 cek
offline) diperluas ke 3 tipe yang belum diuji kausal:
EDUCATIONxINCOME, RACExPOLPARTY, RACExPOLIDEOLOGY.

Bukan kerjaan wajib buat Paper 1 (keputusan desain, bukan lubang), tapi
murah (reuse mesin yang sudah teruji) jadi dikerjakan sekalian.


## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. Sesi bekas crash -> RESTART SESSION.
3. **Download setelah selesai** dari `/kaggle/working/tahap2_probefix_sweep6/`:
   `probe_features_v2.csv`, `probe_features_v2.npz` (Part A),
   `sweep6_rows.csv`, `sweep6_summary_raw.csv` (Part B) -> taruh di
   `notebooks/output/14_tahap2_probefix_sweep6_kaggle/`.

Estimasi: load model ~10 menit; Part A (ekstraksi biasa, 2 layer, banyak
soal) ~15-20 menit; Part B (sweep 3 tipe baru, mesin sama notebook 13)
~15-20 menit. Total ~45 menit - 1 jam.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv tidak ketemu.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
ALL_TYPES = ["AGExPOLPARTY", "EDUCATIONxINCOME", "RACExRELIG",
             "RACExPOLPARTY", "RACExPOLIDEOLOGY", "RELIGxPOLPARTY"]
TYPES_ALREADY_SWEPT = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # notebook 13
TYPES_NEW_SWEEP = [t for t in ALL_TYPES if t not in TYPES_ALREADY_SWEPT]  # Part B
MAX_OPTIONS = 6
STAR_LAYER, STAR_HEAD = 11, 16
L1_LAYER = 1  # kandidat baru finding 09 (unit kausal RACE)

# label demografis per tipe, dipakai bikin pertanyaan QA (sama dgn ATTR_LABELS notebook 09)
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}
ATTR_QA = {ty: (f"What is this survey respondent's {l1}?",
               f"What is this survey respondent's {l2}?")
          for ty, (l1, l2) in ATTR_LABELS.items()}

OUT_DIR = "/kaggle/working/tahap2_probefix_sweep6"
os.makedirs(OUT_DIR, exist_ok=True)


## Data & fungsi prompt bersama (dipakai Part A dan B)

`build_prompt` sama persis dengan notebook 12/13 (identitas = jawaban QA
1 token, lalu soal opini, diakhiri `Answer:`).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, v1_opts, v2_opts, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, v1_opts, v1), "", demo_block(q2, v2_opts, v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def identity_positions(prompt, n_blocks=2):
    positions = []
    search_from = 0
    for _ in range(n_blocks):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))
        positions.append(pos)
        search_from = idx + 1
    return positions

def last_token_abs_pos(prompt):
    """Indeks absolut token TERAKHIR (posisi jawaban-opini), dgn BOS.
    seq_len = 1(BOS) + len(tokens tanpa BOS); indeks terakhir = seq_len - 1
    = len(tokens tanpa BOS). Konsisten dgn konvensi identity_positions()."""
    return len(tokenizer.encode(prompt, add_special_tokens=False))

type_opts = {}
for ty in ALL_TYPES:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    type_opts[ty] = dict(cells=cells, v1_opts=v1_opts, v2_opts=v2_opts,
                         q_per_cell=sub.groupby("group_key")["qkey"].apply(set).to_dict())
    print(f"[{ty}] {len(cells)} sel")


## Part A — Data: soal CAKUPAN LUAS (perbaikan finding 10)

Beda dari notebook 13: soal dipilih berdasar **berapa banyak sel yang
jawab**, bukan beda-pendapat-per-pasangan. Ambil semua soal dengan
cakupan >= 60% sel di tipe itu (minimal 10), lalu **diambil acak
maksimal 250 soal per tipe** (`MAX_Q_PER_TYPE`).

Kenapa di-cap: tanpa cap, Part A = 5.855 soal / 131 ribu forward pass
(~4x lebih lama, dan `np.stack` di akhir butuh ~8,6 GB RAM -> rawan OOM
persis di detik terakhir). Dengan cap: 1.500 soal / 33 ribu forward,
n per tipe tetap 4.000-7.200 baris -- jauh lebih dari cukup buat probe
(finding 10 yang gagal cuma punya n=8-13).

Kenapa acak, bukan top-cakupan: soal dengan cakupan tertinggi numpuk di
2-3 wave survei (EDUCATIONxINCOME: top-250 cuma wave 50/54/92), jadi
probe-nya cuma belajar segelintir topik. Sampel acak dari soal yang
sudah lolos ambang kena semua 15 wave dengan cakupan nyaris sama.


In [ ]:
MIN_COVERAGE_FRAC = 0.6
MIN_COVERAGE_ABS = 10
MAX_Q_PER_TYPE = 250   # cap: 5.855 soal (semua) -> 1.500 soal, ~4x lebih cepat.
                       # n per tipe tetap 4.000-7.200 baris (cukup banget buat probe;
                       # finding 10 yang gagal cuma n=8-13). Naikkan kalau GPU longgar.

probeA_plan = {}
rng_q = np.random.default_rng(RANDOM_SEED)
for ty in ALL_TYPES:
    info = type_opts[ty]
    cells = info["cells"]
    n_cell = len(cells)
    min_cov = max(MIN_COVERAGE_ABS, int(np.ceil(MIN_COVERAGE_FRAC * n_cell)))
    min_cov = min(min_cov, n_cell)  # jangan lebih dari jumlah sel

    q_count = {}
    for gk in cells:
        for qk in info["q_per_cell"].get(gk, set()):
            q_count[qk] = q_count.get(qk, 0) + 1
    good_qs = sorted([qk for qk, c in q_count.items() if c >= min_cov])

    # PENTING: cap-nya ACAK, bukan "ambil cakupan tertinggi". Kalau di-sort by
    # coverage lalu diambil top-N, soal yang kepilih numpuk di 2-3 wave doang
    # (dicek: EDUCATIONxINCOME top-250 cuma wave 50/54/92, acak-250 kena semua
    # 15 wave) -- probe-nya jadi keliatan bagus tapi cuma buat segelintir topik.
    if len(good_qs) > MAX_Q_PER_TYPE:
        idx = rng_q.choice(len(good_qs), size=MAX_Q_PER_TYPE, replace=False)
        picked = sorted(good_qs[i] for i in idx)
    else:
        picked = good_qs

    n_rows = sum(q_count[qk] for qk in picked)
    probeA_plan[ty] = dict(cells=cells, good_qs=picked, min_cov=min_cov,
                           n_good_all=len(good_qs), n_rows_est=n_rows)
    print(f"[{ty}] {n_cell} sel, ambang cakupan={min_cov}, "
          f"soal lolos={len(good_qs)} -> dipakai {len(picked)}, "
          f"perkiraan baris={n_rows}")

print("\nTotal perkiraan baris Part A:",
      sum(p["n_rows_est"] for p in probeA_plan.values()))


## Part A — Ekstraksi: L11 & L1 di posisi jawaban-opini, batched per soal

Hanya 2 layer ditangkap (L1, L11) -- jauh lebih ringan dari notebook 13
(yang nangkep semua 32 buat kebutuhan sweep). Tidak ada patching sama
sekali di Part A -- murni ekstraksi, jadi batch bisa 32.

**Hasil disimpan per tipe** (`probeA_<tipe>.csv/.npz`) begitu tipe itu
kelar, baru digabung di akhir. Kalau sesi mati di tengah jalan, tipe
yang sudah selesai tetap aman -- dan kalau cell ini dijalankan ulang,
tipe yang file-nya sudah ada akan di-skip otomatis.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS
ALL_HEADS = list(range(NUM_HEADS))

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

CAPTURE_LAYERS_A = [L1_LAYER, STAR_LAYER]  # Part A: cuma 2 layer, ringan

_donor_capture = {}
_capture_positions = []
_active_patch = {}  # dipakai lagi di Part B

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().half().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (b, pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                if len(heads) == NUM_HEADS:
                    x[b, pos, :] = x[b, pos, :] + alpha * (d - x[b, pos, :])
                else:
                    for h in heads:
                        s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                        x[b, pos, s] = x[b, pos, s] + alpha * (d[s] - x[b, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

# Part A cuma butuh hook di 2 layer; Part B (sweep) butuh semua 32 --
# pasang SEMUA sekarang (murah, hook nganggur kalau nggak dipicu capture/patch)
handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in range(NUM_LAYERS)]
print(f"Hook terpasang di semua {NUM_LAYERS} layer (Part A pakai 2, Part B pakai semua).")

@torch.no_grad()
def forward_batch(prompts, n_opt, capture_positions=None, patch_spec=None):
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    _active_patch.clear()
    sel = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=1).cpu().numpy()


In [ ]:
BATCH_SIZE = 32   # naik dari 16: Part A murni ekstraksi (tanpa patching),
                  # prompt pendek & seragam panjangnya, muat di T4x2.

def _save_type(ty, rows, v11, v1):
    """Simpan hasil SATU tipe. Dipanggil tiap tipe selesai supaya sesi yang
    mati di tengah jalan nggak bikin semuanya hilang."""
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, f"probeA_{ty}.csv"), index=False)
    np.savez_compressed(os.path.join(OUT_DIR, f"probeA_{ty}.npz"),
                        vecs_l11=np.stack(v11), vecs_l1=np.stack(v1))
    print(f"  [simpan] {ty}: {len(rows)} baris -> probeA_{ty}.csv/.npz")

done_types = []
for ty in ALL_TYPES:
    if os.path.exists(os.path.join(OUT_DIR, f"probeA_{ty}.csv")):
        print(f"[skip] {ty} sudah ada hasilnya (hapus file-nya kalau mau ulang).")
        done_types.append(ty)
        continue

    info = type_opts[ty]
    plan_ty = probeA_plan[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    rows_ty, vec11_ty, vec1_ty = [], [], []

    for qk in tqdm(plan_ty["good_qs"], desc=f"probeA {ty}"):
        gks_here = [gk for gk in plan_ty["cells"] if qk in info["q_per_cell"].get(gk, set())]
        if len(gks_here) < 5:
            continue
        prompts = [build_prompt(ty, v1o, v2o, gk, qk) for gk in gks_here]
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]
        if len(set(lens)) != 1:
            continue  # jarang; skip drpd salah posisi
        pos_abs = last_token_abs_pos(prompts[0])
        n_opt = len(qmeta[qk][2])

        for start in range(0, len(gks_here), BATCH_SIZE):
            batch_gks = gks_here[start:start + BATCH_SIZE]
            batch_prompts = prompts[start:start + BATCH_SIZE]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=[pos_abs])
            for b, gk in enumerate(batch_gks):
                if (gk, qk) not in real_resp:
                    continue
                rows_ty.append(dict(ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                    mouth_pred=",".join(f"{x:.6f}" for x in preds[b])))
                # tetap fp16 (hook-nya sudah .half()): 4.096 dim x 2 layer x 33k baris
                # = 0,55 GB. Versi fp32 lama 4,3 GB -> peak RAM ~8,6 GB pas np.stack,
                # rawan OOM justru di detik terakhir.
                vec11_ty.append(_donor_capture[STAR_LAYER][pos_abs][b].numpy())
                vec1_ty.append(_donor_capture[L1_LAYER][pos_abs][b].numpy())

    _save_type(ty, rows_ty, vec11_ty, vec1_ty)
    done_types.append(ty)
    del rows_ty, vec11_ty, vec1_ty
    gc.collect()

# --- gabung semua tipe jadi satu file (per-tipe tetap disimpan sbg cadangan) ---
try:
    parts = [pd.read_csv(os.path.join(OUT_DIR, f"probeA_{ty}.csv")) for ty in done_types]
    probeA_df = pd.concat(parts, ignore_index=True)
    probeA_df.to_csv(os.path.join(OUT_DIR, "probe_features_v2.csv"), index=False)
    z = [np.load(os.path.join(OUT_DIR, f"probeA_{ty}.npz")) for ty in done_types]
    np.savez_compressed(os.path.join(OUT_DIR, "probe_features_v2.npz"),
                        vecs_l11=np.concatenate([x["vecs_l11"] for x in z]),
                        vecs_l1=np.concatenate([x["vecs_l1"] for x in z]))
    print(probeA_df.shape, "-> probe_features_v2.csv/.npz")
    print("Soal unik terpakai per tipe:\n", probeA_df.groupby("ty")["qk"].nunique())
except MemoryError:
    print("Gabung gagal (memori). Nggak apa-apa: pakai file probeA_<tipe>.csv/.npz, "
          "gabungnya di lokal.")


## Part B — Perluas sweep 32-layer ke 3 tipe baru

Mesin PERSIS notebook 13 (sudah lolos dry-run offline): 33 kondisi
(32 layer + 1 kontrol acak) digabung 1 batch per pasangan-soal.


In [ ]:
N_PAIRS = 12
N_QUESTIONS = 20
MIN_SHARED_Q = 20

rng = np.random.default_rng(RANDOM_SEED)
sweepB_plan = {}
for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    cells = info["cells"]
    q_per_cell = info["q_per_cell"]
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    sweepB_plan[ty] = dict(pairs=pairs, pair_questions=pair_questions)
    print(f"[{ty}] {len(pairs)} pasangan")

rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
_forbidden_ctrl = {(STAR_LAYER, STAR_HEAD)}
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden_ctrl:
        break
print("Head kontrol acak:", RAND_HEAD)


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

CAPTURE_LAYERS_B = list(range(NUM_LAYERS))

# Pass 1: baseline + donor semua layer (batched per soal, spt notebook 12 fix)
baselineB_pred = {}
donorsB = {}
id_posB = {}

for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    needed = sorted({(gk, qk) for (a, b), qs in sweepB_plan[ty]["pair_questions"].items()
                     for qk in qs for gk in (a, b)})
    by_qk = {}
    for (gk, qk) in needed:
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baselineB {ty}"):
        prompts = [build_prompt(ty, v1o, v2o, gk, qk) for gk in gks]
        if (ty, qk) not in id_posB:
            id_posB[(ty, qk)] = identity_positions(prompts[0])
        positions = id_posB[(ty, qk)]
        n_opt = len(qmeta[qk][2])
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]
        if len(set(lens)) != 1:
            print(f"  [{ty}/{qk}] panjang tidak seragam -> skip")
            continue
        for start in range(0, len(gks), 16):
            batch_gks = gks[start:start + 16]
            batch_prompts = prompts[start:start + 16]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=positions)
            for b, gk in enumerate(batch_gks):
                baselineB_pred[(gk, qk)] = preds[b]
                donorsB[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                    for L in CAPTURE_LAYERS_B
                }
print(f"{len(baselineB_pred)} baseline Part B selesai.")


In [ ]:
sweepB_rows = []
for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    for (A, B) in tqdm(sweepB_plan[ty]["pairs"], desc=f"sweepB {ty}"):
        for qk in sweepB_plan[ty]["pair_questions"][(A, B)]:
            if (ty, qk) not in id_posB or (ty, A, qk) not in donorsB or (ty, B, qk) not in donorsB:
                continue
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            positions = id_posB[(ty, qk)]
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baselineB_pred[(A, qk)], baselineB_pred[(B, qk)]
            prompt_A = build_prompt(ty, v1o, v2o, A, qk)

            n_cond = NUM_LAYERS + 1
            batch_prompts = [prompt_A] * n_cond
            spec = {}
            for L in range(NUM_LAYERS):
                spec.setdefault(L, [])
                for pos in positions:
                    spec[L].append((L, pos, ALL_HEADS, 1.0, donorsB[(ty, B, qk)][L][pos]))
            ctrl_idx = NUM_LAYERS
            rl, rh = RAND_HEAD
            spec.setdefault(rl, [])
            for pos in positions:
                spec[rl].append((ctrl_idx, pos, [rh], 1.0, donorsB[(ty, B, qk)][rl][pos]))

            preds = forward_batch(batch_prompts, n_opt, patch_spec=spec)

            wd_ctrl_realB = wd(preds[ctrl_idx], realB, ordinal)
            wd_A_realA = wd(predA, realA, ordinal)
            wd_A_realB = wd(predA, realB, ordinal)
            wd_B_realB = wd(predB, realB, ordinal)
            for L in range(NUM_LAYERS):
                sweepB_rows.append(dict(
                    attr_type=ty, pair=f"{A} -> {B}", qkey=qk, layer=L,
                    wd_A_to_realA=wd_A_realA, wd_A_to_realB=wd_A_realB,
                    wd_B_to_realB=wd_B_realB,
                    wd_patch_to_realB=wd(preds[L], realB, ordinal),
                    wd_ctrl_to_realB=wd_ctrl_realB,
                ))

sweepB = pd.DataFrame(sweepB_rows)
sweepB["shift_ke_realB"] = sweepB["wd_A_to_realB"] - sweepB["wd_patch_to_realB"]
sweepB["shift_ctrl_ke_realB"] = sweepB["wd_A_to_realB"] - sweepB["wd_ctrl_to_realB"]
sweepB.to_csv(os.path.join(OUT_DIR, "sweep6_rows.csv"), index=False)
print(sweepB.shape, "-> sweep6_rows.csv (3 tipe baru; koreksi max-stat permutation dikerjakan lokal)")


## Cara baca hasil & checklist

**Part A (probe):** setelah download, fitting probe lokal (ridge +
LOOCV) dari `probe_features_v2.csv/.npz` -- sama seperti sebelumnya,
tapi sekarang tiap soal punya cakupan sel jauh lebih luas. Bandingkan
juga vektor L1 vs L11 -- apakah RACE terbaca lebih baik dari L1?
Vektornya fp16; cast ke float32 dulu sebelum fitting.

**Yang dibutuhkan paper dari Part A** (§6.5 "How much is lost in the
corridor" + §8): satu angka per tipe -- skor probe dari L11/L1 vs skor
"mulut" model, dua-duanya dilawan ground truth survei. Ekspektasinya
bergradasi ngikutin tabel kesetiaan (finding 06).

**Part B (sweep 3 tipe baru):** koreksi max-statistic permutation
(2000x, sama seperti finding 09) dikerjakan LOKAL dari `sweep6_rows.csv`
-- cari layer mana yang signifikan per tipe, sandingkan dengan hasil
finding 06 (peta kesetiaan 6 tipe) buat lihat pola "kesetiaan tinggi ->
kausal ikut kuat" berlaku di 3 tipe ini juga atau tidak.

**Download (WAJIB):** `probe_features_v2.csv`, `probe_features_v2.npz`,
`sweep6_rows.csv` -> `notebooks/output/14_tahap2_probefix_sweep6_kaggle/`.
(Kalau sesi kepotong: `probeA_<tipe>.csv/.npz` juga cukup, tinggal
digabung di lokal.)
